# Tarea 03 — Generación de Imágenes con Autoencoder
**Pytorch Lightning · Hydra · WandB**

**Curso:** Inteligencia Artificial — Escuela de Ingeniería en Computación, TEC

---

### Objetivo
Implementar y comparar dos arquitecturas de autoencoder para reconstrucción de imágenes del dataset MVTec AD:
- **VAE** (Variational Autoencoder)
- **U-Net AE** (Autoencoder con skip connections)

Cada arquitectura se entrena con **4 funciones de pérdida**: L1, L2, SSIM, SSIM+L1, totalizando **8 experimentos**.

### Dataset
MVTec AD — 4 clases industriales: `cable`, `capsule`, `screw`, `transistor`.  
Imágenes de 128×128 con 3 canales (RGB).

### Herramientas
- **PyTorch Lightning** → estructura del entrenamiento
- **Hydra** → gestión modular de configuraciones
- **WandB** → logging, comparación de métricas y visualizaciones

---
## 1. Configuración del entorno
Instalación de dependencias necesarias.

In [ ]:
!pip install -q \
    pytorch-lightning>=2.0.0 \
    hydra-core>=1.3.0 \
    wandb \
    pytorch-msssim \
    scikit-learn \
    omegaconf

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA disponible:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

---
## 2. Clonar repositorio
Se clona el repositorio con el código fuente del proyecto.

In [ ]:
import os

REPO_URL = 'https://github.com/AllanDBB/mvtec-autoencoder-reconstruction.git'
REPO_DIR = '/content/mvtec-autoencoder-reconstruction'

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print('Directorio de trabajo:', os.getcwd())

---
## 3. Descarga del Dataset MVTec AD

Se descargan únicamente las 4 clases requeridas: `cable`, `capsule`, `screw`, `transistor`.

> **Opción A (Kaggle API)** — recomendado.  
> **Opción B (descarga directa)** — en caso de que Kaggle no esté disponible.

In [ ]:
import os, tarfile, shutil
from pathlib import Path
from getpass import getpass

# ── Credenciales Kaggle ───────────────────────────────────────────────────────
# Entra a https://www.kaggle.com → Account → API → "Create New Token"
# Te descarga un kaggle.json con username y key
os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY']      = getpass('Kaggle API key : ')

# ── Descarga ──────────────────────────────────────────────────────────────────
DATA_DIR = '/content/mvtec'
os.makedirs(DATA_DIR, exist_ok=True)

!pip install -q kaggle
!kaggle datasets download -d ipythonx/mvtec-ad -p /tmp/mvtec_zip --unzip -q

# Copia solo las 4 clases necesarias
CLASSES = ['cable', 'capsule', 'screw', 'transistor']
for cls in CLASSES:
    src = Path(f'/tmp/mvtec_zip/{cls}')
    dst = Path(f'{DATA_DIR}/{cls}')
    if src.exists() and not dst.exists():
        shutil.copytree(src, dst)
        print(f'  {cls}: copiado')
    elif dst.exists():
        print(f'  {cls}: ya existe')
    else:
        print(f'  {cls}: no encontrado en /tmp/mvtec_zip — revisa la estructura del zip')

print('\nDataset listo en:', DATA_DIR)

In [ ]:
# Verificación rápida del dataset descargado
for cls in CLASSES:
    train_good = list(Path(f'{DATA_DIR}/{cls}/train/good').glob('*.png'))
    test_dirs  = list(Path(f'{DATA_DIR}/{cls}/test').iterdir())
    print(f'{cls}: {len(train_good)} train/good | {len(test_dirs)} categorías de test')

### 3.1 Verificación del dataset

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

from src.datasets.mvtec import MVTecDataModule

dm = MVTecDataModule(
    data_dir=DATA_DIR,
    classes=['cable', 'capsule', 'screw', 'transistor'],
    image_size=128,
    batch_size=32,
    num_workers=2,
)
dm.setup()

print(f'Train samples : {len(dm.train_dataset)}')
print(f'Val   samples : {len(dm.val_dataset)}')
print(f'Test  samples : {len(dm.test_dataset)}')

In [ ]:
import matplotlib.pyplot as plt

train_dl = dm.train_dataloader()
batch = next(iter(train_dl))
imgs, labels, defect_types, classes = batch

fig, axes = plt.subplots(2, 8, figsize=(16, 4))
for i in range(8):
    axes[0, i].imshow(imgs[i].permute(1, 2, 0).clamp(0, 1))
    axes[0, i].set_title(f'{classes[i]}', fontsize=8)
    axes[0, i].axis('off')
for i in range(8):
    axes[1, i].imshow(imgs[8 + i].permute(1, 2, 0).clamp(0, 1))
    axes[1, i].set_title(f'{classes[8 + i]}', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Ejemplos del set de entrenamiento (imágenes sin defectos)')
plt.tight_layout()
plt.show()

---
## 4. Arquitecturas de los Modelos

### VAE (Variational Autoencoder)
- **Encoder**: 5 capas Conv2d con stride 2 → 128×128 → 4×4 → espacio latente μ, σ
- **Reparametrización**: z = μ + ε·σ
- **Decoder**: 5 capas ConvTranspose2d → 4×4 → 128×128
- **Pérdida**: Reconstrucción + β·KL-divergencia

### U-Net AE (Autoencoder con Skip Connections)
- **Encoder**: 4 bloques Conv2d con stride 2, guardando skip connections
- **Cuello de botella**: Conv adicional → proyección lineal a latent_dim (para t-SNE)
- **Decoder**: ConvTranspose2d + concatenación de skip connections
- Las skip connections permiten recuperar detalles finos perdidos en la compresión

In [ ]:
from src.models.vae import VAE
from src.models.unet_ae import UNetAE

vae = VAE(latent_dim=128)
unet = UNetAE(latent_dim=128)

dummy = torch.randn(2, 3, 128, 128)

recon_vae, mu, logvar = vae(dummy)
print(f'VAE   — input: {dummy.shape} | recon: {recon_vae.shape} | mu: {mu.shape}')

recon_un, z = unet(dummy)
print(f'UNetAE — input: {dummy.shape} | recon: {recon_un.shape} | z: {z.shape}')

def count_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

print(f'\nParámetros VAE   : {count_params(vae):,}')
print(f'Parámetros UNetAE: {count_params(unet):,}')

---
## 5. Configuración de WandB
Login necesario para registrar los experimentos.

In [ ]:
import wandb

# ── Opción A (RECOMENDADA): Colab Secrets ────────────────────────────────────
# 1. Abre el panel izquierdo de Colab → ícono de llave 🔑 "Secrets"
# 2. Crea un secret llamado  WANDB_API_KEY  y pega tu clave ahí
# 3. Activa el toggle "Notebook access"
# Luego descomenta las dos líneas siguientes:
# from google.colab import userdata
# wandb.login(key=userdata.get('WANDB_API_KEY'))

# ── Opción B: pegar la clave directamente (NO hacer git push con esto) ────────
wandb.login(key="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")

---
## 6. Entrenamiento de Experimentos

Se ejecutan **8 experimentos** en total:
- 4 para el VAE: pérdidas L1, L2, SSIM, SSIM+L1
- 4 para el U-Net AE: mismas pérdidas

Cada experimento se controla con Hydra pasando overrides desde la línea de comandos.  
Los resultados se registran automáticamente en WandB.

> **Nota**: cada experimento tarda ~10-20 min en Colab con GPU T4 (50 épocas).  
> Puedes reducir `trainer.max_epochs=20` para pruebas rápidas.

In [ ]:
# === VAE — 4 funciones de pérdida ===
EPOCHS = 50

for loss_fn in ['l1', 'l2', 'ssim', 'ssim_l1']:
    print(f'\n{"="*60}')
    print(f'  VAE  |  Loss: {loss_fn.upper()}  |  Epochs: {EPOCHS}')
    print(f'{"="*60}\n')
    !python {REPO_DIR}/train.py \
        model=vae \
        model.loss={loss_fn} \
        trainer.max_epochs={EPOCHS} \
        data.data_dir={DATA_DIR} \
        --config-dir={REPO_DIR}/conf

In [ ]:
# === U-Net AE — 4 funciones de pérdida ===
for loss_fn in ['l1', 'l2', 'ssim', 'ssim_l1']:
    print(f'\n{"="*60}')
    print(f'  UNetAE  |  Loss: {loss_fn.upper()}  |  Epochs: {EPOCHS}')
    print(f'{"="*60}\n')
    !python {REPO_DIR}/train.py \
        model=u-net \
        model.loss={loss_fn} \
        trainer.max_epochs={EPOCHS} \
        data.data_dir={DATA_DIR} \
        --config-dir={REPO_DIR}/conf

---
## 7. Resultados de WandB

Las siguientes métricas y visualizaciones se registraron automáticamente en WandB durante el entrenamiento:

| Métrica | Descripción |
|---|---|
| `train/loss` | Pérdida de entrenamiento por paso/época |
| `val/loss` | Pérdida de validación por época |
| `val/reconstructions` | Grid 16 imágenes: original vs reconstruida (val) |
| `test/tsne` | t-SNE del espacio latente en test (Good vs Defective) |
| `test/reconstructions_good_bad` | Grid: imágenes buenas y defectuosas reconstruidas |

### 7.1 Gráficas de pérdida — Insertar desde WandB
> Descarga las gráficas desde tu proyecto WandB e insértalas aquí.

In [ ]:
# Descarga resumen de runs desde WandB para análisis local
import wandb
import pandas as pd

WANDB_PROJECT = 'mvtec-autoencoder-reconstruction'
WANDB_ENTITY  = None  # Tu entity de WandB (o None para usar el default)

api = wandb.Api()
runs = api.runs(f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT)

records = []
for run in runs:
    records.append({
        'name': run.name,
        'state': run.state,
        'val_loss_best': run.summary.get('val/loss', None),
        'model': run.config.get('model', {}).get('name', ''),
        'loss_fn': run.config.get('model', {}).get('loss', ''),
        'latent_dim': run.config.get('model', {}).get('latent_dim', ''),
        'epochs': run.config.get('trainer', {}).get('max_epochs', ''),
    })

df_runs = pd.DataFrame(records)
df_runs = df_runs.sort_values(['model', 'loss_fn'])
print(df_runs.to_string(index=False))

---
## 8. Evaluación del Modelo

Para la evaluación se carga el mejor checkpoint de cada configuración y se calcula el **error de reconstrucción** sobre el set de prueba.  
Se compara el error entre imágenes **normales** y **defectuosas** para detectar anomalías.

In [ ]:
import glob
import torch
import numpy as np
from src.lightning_module import AutoencoderModule
from src.datasets.mvtec import MVTecDataModule

# Busca todos los checkpoints guardados por Hydra
ckpts = sorted(glob.glob(f'{REPO_DIR}/outputs/**/*.ckpt', recursive=True))
print(f'Checkpoints encontrados: {len(ckpts)}')
for c in ckpts:
    print(' ', c)

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

dm_eval = MVTecDataModule(
    data_dir=DATA_DIR,
    classes=['cable', 'capsule', 'screw', 'transistor'],
    image_size=128,
    batch_size=64,
    num_workers=2,
)
dm_eval.setup()
test_loader = dm_eval.test_dataloader()

all_results = {}  # key = run_name

for ckpt_path in ckpts:
    run_name = ckpt_path.split('outputs/')[-1].split('/')[0]  # fecha/hora_model_loss
    print(f'\nEvaluando: {run_name}')

    try:
        module = AutoencoderModule.load_from_checkpoint(ckpt_path, map_location=device)
        module.eval().to(device)
    except Exception as e:
        print(f'  Error al cargar: {e}')
        continue

    errors, labels_list, defect_list, class_list = [], [], [], []
    with torch.no_grad():
        for batch in test_loader:
            imgs, lbl, defect_types, classes = batch
            imgs = imgs.to(device)
            if module.is_vae:
                recon, mu, logvar = module.model(imgs)
            else:
                recon, _ = module.model(imgs)
            err = torch.mean((recon - imgs) ** 2, dim=[1, 2, 3]).cpu().numpy()
            errors.extend(err)
            labels_list.extend(lbl.numpy())
            defect_list.extend(list(defect_types))
            class_list.extend(list(classes))

    all_results[run_name] = {
        'errors': np.array(errors),
        'labels': np.array(labels_list),
        'defects': defect_list,
        'classes': class_list,
        'model': module.model_cfg.name,
        'loss_fn': module.model_cfg.loss,
    }
    print(f'  Samples evaluados: {len(errors)}')

print('\nEvaluación completa.')

### 8.1 Histogramas de Error de Reconstrucción

Para cada configuración, se visualiza la distribución del error de reconstrucción separando imágenes **normales (Good)** de **defectuosas (Defective)**.  
Un buen modelo debería mostrar una separación clara entre las dos distribuciones: bajo error para imágenes normales, alto error para defectuosas.

In [ ]:
import matplotlib.pyplot as plt

n_runs = len(all_results)
fig, axes = plt.subplots(2, n_runs // 2 + n_runs % 2, figsize=(5 * (n_runs // 2 + n_runs % 2), 8))
axes = axes.flatten()

for ax, (run_name, res) in zip(axes, all_results.items()):
    good_err = res['errors'][res['labels'] == 0]
    bad_err  = res['errors'][res['labels'] == 1]

    bins = np.linspace(0, max(res['errors'].max(), 0.01), 40)
    ax.hist(good_err, bins=bins, alpha=0.6, color='green', label='Normal', density=True)
    ax.hist(bad_err,  bins=bins, alpha=0.6, color='red',   label='Defectuosa', density=True)
    ax.set_title(f"{res['model'].upper()} — {res['loss_fn'].upper()}", fontsize=10)
    ax.set_xlabel('Error de reconstrucción (MSE)')
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=8)

for ax in axes[n_runs:]:
    ax.set_visible(False)

plt.suptitle('Histogramas de Error de Reconstrucción — Normal vs Defectuosa', fontsize=13)
plt.tight_layout()
plt.savefig('histogramas_error.png', dpi=120, bbox_inches='tight')
plt.show()
print('Guardado: histogramas_error.png')

### 8.2 Error por Subclase de Defecto

Se desglosa el error de reconstrucción por cada tipo específico de defecto dentro del set de prueba.  
Esto permite identificar qué tipos de anomalías el modelo detecta mejor o peor.

In [ ]:
# Tomar el mejor run por modelo (menor val loss) — ajustar keys si es necesario
# Aquí mostramos el primer run disponible a modo de ejemplo
run_name, res = next(iter(all_results.items()))
errors = res['errors']
defects = np.array(res['defects'])
labels = res['labels']

unique_defects = sorted(set(defects))
n_d = len(unique_defects)

fig, axes = plt.subplots(1, n_d, figsize=(4 * n_d, 4), sharey=False)
if n_d == 1:
    axes = [axes]

for ax, defect in zip(axes, unique_defects):
    mask = defects == defect
    err_subset = errors[mask]
    color = 'green' if defect == 'good' else 'red'
    ax.hist(err_subset, bins=30, color=color, alpha=0.75, density=True)
    ax.set_title(defect, fontsize=9)
    ax.set_xlabel('MSE')
    ax.set_ylabel('Densidad')
    mean_err = err_subset.mean()
    ax.axvline(mean_err, color='black', linestyle='--', linewidth=1, label=f'μ={mean_err:.4f}')
    ax.legend(fontsize=7)

plt.suptitle(f'Error por tipo de defecto — {res["model"].upper()} ({res["loss_fn"].upper()})')
plt.tight_layout()
plt.savefig('error_por_defecto.png', dpi=120, bbox_inches='tight')
plt.show()

---
## 9. Comparación de Configuraciones

### 9.1 Tabla resumen de pérdidas de validación

Se comparan las métricas finales de validación entre las 8 configuraciones.

In [ ]:
import pandas as pd

summary = []
for run_name, res in all_results.items():
    good_err = res['errors'][res['labels'] == 0]
    bad_err  = res['errors'][res['labels'] == 1]
    summary.append({
        'Modelo': res['model'].upper(),
        'Pérdida': res['loss_fn'].upper(),
        'MSE Normal (μ)': f"{good_err.mean():.5f}",
        'MSE Normal (σ)': f"{good_err.std():.5f}",
        'MSE Defecto (μ)': f"{bad_err.mean():.5f}",
        'MSE Defecto (σ)': f"{bad_err.std():.5f}",
        'Separación (Δμ)': f"{bad_err.mean() - good_err.mean():.5f}",
    })

df_summary = pd.DataFrame(summary).sort_values(['Modelo', 'Pérdida'])
df_summary

### 9.2 Comparación visual: reconstrucciones correctas vs defectuosas

Se comparan imágenes reconstruidas por el mejor VAE y el mejor U-Net AE, mostrando lado a lado imágenes normales y defectuosas.  
El modelo debería reconstruir bien las imágenes normales (error bajo) pero presentar error notable en las defectuosas.

In [ ]:
def show_reconstructions(ckpt_path, title, n_good=8, n_bad=8):
    module = AutoencoderModule.load_from_checkpoint(ckpt_path, map_location=device)
    module.eval().to(device)

    good_imgs, good_recons = [], []
    bad_imgs,  bad_recons  = [], []

    with torch.no_grad():
        for batch in test_loader:
            imgs, lbl, _, _ = batch
            imgs = imgs.to(device)
            recon = module.model(imgs)[0] if module.is_vae else module.model(imgs)[0]
            for i in range(len(imgs)):
                if lbl[i] == 0 and len(good_imgs) < n_good:
                    good_imgs.append(imgs[i].cpu())
                    good_recons.append(recon[i].cpu())
                elif lbl[i] == 1 and len(bad_imgs) < n_bad:
                    bad_imgs.append(imgs[i].cpu())
                    bad_recons.append(recon[i].cpu())
            if len(good_imgs) >= n_good and len(bad_imgs) >= n_bad:
                break

    all_orig  = good_imgs[:n_good] + bad_imgs[:n_bad]
    all_recon = good_recons[:n_good] + bad_recons[:n_bad]
    n = len(all_orig)

    fig, axes = plt.subplots(2, n, figsize=(n * 1.8, 4))
    for i in range(n):
        axes[0, i].imshow(all_orig[i].permute(1, 2, 0).clamp(0, 1))
        axes[0, i].axis('off')
        border_color = 'green' if i < n_good else 'red'
        for spine in axes[0, i].spines.values():
            spine.set_edgecolor(border_color)
            spine.set_linewidth(2)
        axes[1, i].imshow(all_recon[i].permute(1, 2, 0).clamp(0, 1))
        axes[1, i].axis('off')

    axes[0, 0].set_ylabel('Original', fontsize=9)
    axes[1, 0].set_ylabel('Reconstruida', fontsize=9)
    axes[0, 0].annotate('Normal →', xy=(0, 0.5), xycoords='axes fraction',
                        fontsize=7, color='green', va='center')
    axes[0, n_good].annotate('← Defecto', xy=(0, 0.5), xycoords='axes fraction',
                              fontsize=7, color='red', va='center')
    plt.suptitle(title)
    plt.tight_layout()
    safe_title = title.replace(' ', '_').replace('|', '').replace('(', '').replace(')', '')
    plt.savefig(f'recon_{safe_title}.png', dpi=120, bbox_inches='tight')
    plt.show()


# Mostrar el primer y segundo checkpoint disponibles como ejemplo
if len(ckpts) >= 1:
    show_reconstructions(ckpts[0], f'Modelo 1 — {all_results[list(all_results.keys())[0]]["model"].upper()} | {all_results[list(all_results.keys())[0]]["loss_fn"].upper()}')
if len(ckpts) >= 2:
    show_reconstructions(ckpts[1], f'Modelo 2 — {all_results[list(all_results.keys())[1]]["model"].upper()} | {all_results[list(all_results.keys())[1]]["loss_fn"].upper()}')

---
## 10. Análisis Crítico de Resultados

### 10.1 Comparación de funciones de pérdida

**L1 vs L2:**  
- L2 penaliza fuertemente los errores grandes (cuadrático), lo que puede causar *blurriness* (reconstrucciones borrosas).  
- L1 es más robusta a outliers y tiende a preservar mejor los bordes.

**SSIM:**  
- Mide la similitud estructural (brillo, contraste, estructura). Modela mejor la percepción humana de calidad de imagen.  
- Puede ser inestable en etapas tempranas del entrenamiento cuando las reconstrucciones son muy malas.

**SSIM + L1:**  
- Combinación que aprovecha la percepción estructural de SSIM y la estabilidad de L1.  
- En la práctica suele dar las mejores reconstrucciones visuales.

### 10.2 Comparación VAE vs U-Net AE

**VAE:**  
- Espacio latente regularizado (distribución gaussiana) → representación más suave y continua.  
- La pérdida KL puede entrar en conflicto con la calidad de reconstrucción (trade-off β).  
- El vector latente μ tiene interpretación probabilística clara para t-SNE.

**U-Net AE:**  
- Las skip connections permiten recuperar detalles finos (texturas, bordes) perdidos en la compresión.  
- Reconstrucciones generalmente más nítidas, especialmente para imágenes sin defectos.  
- Para la detección de anomalías, esto puede ser una desventaja: si el modelo reconstruye bien TAMBIÉN las anomalías, el error no las detecta.

### 10.3 t-SNE del espacio latente

El análisis t-SNE muestra cómo el modelo organiza las representaciones internas:  
- Un modelo bien entrenado debería agrupar las imágenes normales en clusters compactos.  
- Las imágenes defectuosas deberían aparecer en regiones distintas o más dispersas del espacio latente.  
- A lo largo de las épocas se observa cómo los clusters se van formando progresivamente.

> *(Insertar aquí las imágenes de t-SNE descargadas de WandB)*

---
## 11. Imágenes de WandB

Descarga y muestra las imágenes registradas en WandB directamente en el notebook.

In [ ]:
from IPython.display import Image, display
import wandb

api = wandb.Api()
project_path = f"{WANDB_ENTITY}/{WANDB_PROJECT}" if WANDB_ENTITY else WANDB_PROJECT
runs = api.runs(project_path)

os.makedirs('/content/wandb_imgs', exist_ok=True)

for run in runs:
    print(f'\n=== Run: {run.name} ===')
    for file in run.files():
        if file.name.endswith('.png') or 'media/images' in file.name:
            local_path = f'/content/wandb_imgs/{run.name}_{os.path.basename(file.name)}'
            file.download(root='/content/wandb_imgs', replace=True)
            print(f'  {file.name}')

---
## 12. Conclusiones

*(Completar con los resultados reales de los experimentos)*

1. **Mejor función de pérdida para VAE**: `___` — razón: ...
2. **Mejor función de pérdida para U-Net AE**: `___` — razón: ...
3. **Mejor arquitectura general**: `___` — con pérdida `___`
4. **Detección de anomalías**: El error de reconstrucción resultó ser `[efectivo / poco efectivo]` como score de detección porque ...
5. **Separación en espacio latente**: El t-SNE mostró `[buena / moderada / poca]` separación entre clases normales y defectuosas ...

---
*Tarea 03 — Inteligencia Artificial — Instituto Tecnológico de Costa Rica*